In [ ]:
! pip install transformers evaluate datasets requests pandas imbalanced-learn sklearn

In [ ]:
from huggingface_hub import notebook_login
from transformers import AutoTokenizer
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from imblearn.over_sampling import RandomOverSampler

In [ ]:
notebook_login()


In [ ]:
!apt install git-lfs

In [ ]:
model_checkpoint = "facebook/esm2_t12_35M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/seq_training_data.csv')

In [ ]:
sequences = df['sequence'].tolist()
labels = df['type'].tolist()

In [ ]:
SEED = 41

In [ ]:
# Get sequences and labels
sequences = df['sequence'].tolist()
labels = df['type'].tolist()

# First split: train/test
train_sequences, test_sequences, train_labels, test_labels = train_test_split(
    sequences, labels, 
    test_size=0.2,
    stratify=labels,  # FIX: stratify on labels directly
    random_state=SEED
)

# Second split: train/val
train_sequences, val_sequences, train_labels, val_labels = train_test_split(
    train_sequences, train_labels, 
    test_size=0.2,
    stratify=train_labels,  # FIX: stratify on train_labels directly
    random_state=SEED
)

print(f"Before oversampling - Train: {len(train_sequences)}, Class distribution: {np.bincount(train_labels)}")

# Oversample training data (FIX: need to reshape for RandomOverSampler)
train_sequences_array = np.array(train_sequences).reshape(-1, 1)
ros = RandomOverSampler(random_state=SEED)
X_train_resampled, y_train_resampled = ros.fit_resample(train_sequences_array, train_labels)

# Convert back to list
train_sequences_resampled = X_train_resampled.flatten().tolist()
train_labels_resampled = y_train_resampled.tolist()

print(f"After oversampling - Train: {len(train_sequences_resampled)}, Class distribution: {np.bincount(train_labels_resampled)}")

# Tokenize with RESAMPLED data (FIX: use resampled sequences and labels)
train_tokenized = tokenizer(train_sequences_resampled, truncation=True, padding='max_length', max_length=512)
val_tokenized = tokenizer(val_sequences, truncation=True, padding='max_length', max_length=512)

# Create datasets
train_dataset = Dataset.from_dict(train_tokenized)
val_dataset = Dataset.from_dict(val_tokenized)

# Add labels (FIX: use resampled labels for training)
train_dataset = train_dataset.add_column("labels", train_labels_resampled)
val_dataset = val_dataset.add_column("labels", val_labels)
